In [4]:
import torch
import torch.nn as nn

In [ ]:
# Input shape:  (N, C)
# Output shape: (N, C)
# Where N is the mini-batch size and C is the number of features.
class BatchNorm1d(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.num_features = num_features

        self.running_variance = torch.zeros(num_features)
        self.running_mean = torch.zeros(num_features)

        self.gamma = nn.Parameter(torch.ones(num_features))
        self.betta = nn.Parameter(torch.zeros(num_features))

        # Default used by pytorch
        self.epsilon = 1e-5

    def forward(self, x: torch.Tensor):
        N, C = x.shape
        assert C == self.num_features

        mini_batch = x
        if self.training:
            current_variance_per_feature, current_mean_per_feature = torch.var_mean(mini_batch, dim=0)
            self.running_variance = (self.running_variance + current_variance_per_feature) / 2
            self.running_mean = (self.running_mean + current_mean_per_feature) / 2
            variance_to_use, mean_to_use = current_variance_per_feature, current_mean_per_feature
        else:
            variance_to_use, mean_to_use = self.running_variance, self.running_mean

        # Epsilon added for numerical stability
        x = (x - mean_to_use) / torch.sqrt(variance_to_use + self.epsilon)

        # Allow the model to learn a different distribution if needed
        x = x * self.gamma + self.betta

        return x


# bn = BatchNorm1d(num_features=2)
# y = torch.tensor([[2, 4], [5, 6]])
# y_hat = bn.forward(torch.tensor([[1, 2], [3, 4]], dtype=torch.float32))


# Input shape:  (N, C, H, W)
# Output shape: (N, C, H, W)
# Where N is the mini-batch size, C is the number of channels and H and W are the width and height of the channels.
class BatchNorm2d(nn.Module):
    def __init__(self, num_channels):
        super().__init__()
        self.num_channels = num_channels

        self.running_variance = torch.zeros(num_channels)
        self.running_mean = torch.zeros(num_channels)

        self.gamma = nn.Parameter(torch.ones(num_channels))
        self.betta = nn.Parameter(torch.zeros(num_channels))

        # Default used by pytorch
        self.epsilon = 1e-5

    def forward(self, x: torch.Tensor):
        N, C, H, W = x.shape
        assert C == self.num_channels

        mini_batch = x
        if self.training:
            # Calculate mean per channel, so we calculate it OVER the mini bach the width and the height.
            # The mini-batch is the 0th dimension, the 1st is the channel (so this one we want to preserve), then the 2nd and 3rd is the height and width respectively
            current_variance_per_channel, current_mean_per_channel = torch.var_mean(mini_batch, dim=[0, 2, 3])

            self.running_variance = (self.running_variance + current_variance_per_channel) / 2
            self.running_mean = (self.running_mean + current_mean_per_channel) / 2
            variance_to_use, mean_to_use = current_variance_per_channel, current_mean_per_channel
        else:
            variance_to_use, mean_to_use = self.running_variance, self.running_mean

        # Reshape both mean and variance to affect only the desired channel
        mean_to_use = mean_to_use.view(1, self.num_channels, 1, 1)
        variance_to_use = variance_to_use.view(1, self.num_channels, 1, 1)

        # Epsilon added for numerical stability
        x = (x - mean_to_use) / torch.sqrt(variance_to_use + self.epsilon)

        # Allow the model to learn a different distribution if needed
        x = x * self.gamma + self.betta

        return x


# image1 = [
#     # Chan1 for example red
#     [
#         [1, 2],
#         [4, 5],
#     ],
#     # Chan2 for example blue
#     [
#         [4, 5],
#         [5, 123],
#     ],
# ]

# image2 = [
#     [
#         [5, 6],
#         [2, 7],
#     ],
#     [
#         [2, 46],
#         [234, 27],
#     ],
# ]

# mini_batch = torch.tensor([image1, image2], dtype=torch.float32)
# bn = BatchNorm2d(num_channels=2)
# y_hat = bn.forward(mini_batch)

tensor([[[[-1.4031, -0.9354],
          [ 0.0000,  0.4677]],

         [[-0.6254, -0.6133],
          [-0.6133,  0.8127]]],


        [[[ 0.4677,  0.9354],
          [-0.9354,  1.4031]],

         [[-0.6495, -0.1178],
          [ 2.1540, -0.3474]]]], grad_fn=<AddBackward0>)